<a href="https://colab.research.google.com/github/shoh0806/Capstone_Design/blob/main/MF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#0. File upload


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ratings.csv to ratings.csv


#1. import

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import optim
import math
import random
import time
import os

#2 SEED 고정

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device: {device}")

Device: cpu


#3. 하이퍼파라미터

In [ ]:
DATA_DIR     = "/content"
hidden_dim   = 100
batch_size   = 128
LR           = 0.001
weight_decay = 0.0001
epochs       = 30
NEG_EVAL     = 100
patience     = 10

#4. 데이터 전처리 및 데이터 분리

In [ ]:
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
ratings = ratings.sort_values(by=["userId", "timestamp"])

user_seq = ratings.groupby("userId")["movieId"].apply(list)

item_set = ratings["movieId"].unique()
item2idx = {item: i+1 for i, item in enumerate(item_set)}

lengths = user_seq.apply(len)
print(f"평균: {lengths.mean():.1f}  최대: {lengths.max()}  최소: {lengths.min()}  중앙값: {lengths.median()}")

user_sequences = [[item2idx[i] for i in seq] for seq in user_seq]

# Train / Val / Test 분리 (leave-two-out)
train_sequences = []
val_data        = []
test_data       = []

for seq in user_sequences:
    if len(seq) < 20:
        continue
    train = seq[:-2]
    val   = seq[-2]
    test  = seq[-1]

    train_sequences.append(train)
    val_data.append((train, [val]))
    test_data.append((train + [val], [test]))

print(f"학습 유저 수: {len(train_sequences):,}")

all_item_list = list(item2idx.values())

평균: 165.6  최대: 2314  최소: 20  중앙값: 96.0
학습 유저 수: 6,040


#5. Dataset(MF)


In [ ]:
class MFDataset(Dataset):
    def __init__(self, sequences, all_items):
        self.data = []
        for user_id, seq in enumerate(sequences):
            seq_set = set(seq)
            for item in seq:
                neg = random.choice(all_items)
                while neg in seq_set:
                    neg = random.choice(all_items)
                self.data.append((user_id, item, neg))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user_id, pos, neg = self.data[idx]
        return (torch.LongTensor([user_id]),
                torch.LongTensor([pos]),
                torch.LongTensor([neg]))

dataset = MFDataset(train_sequences, all_item_list)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"학습 샘플 수: {len(dataset):,}")

학습 샘플 수: 988,129


#6. MF Model

In [ ]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, hidden_dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_users + 1, hidden_dim)
        self.item_emb = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)

    def forward(self, user, item):
        u = self.user_emb(user)   # (B, D)
        i = self.item_emb(item)   # (B, D)
        return (u * i).sum(-1)    # (B,) 내적

num_users = len(train_sequences)
model     = MF(num_users, len(item2idx), hidden_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=weight_decay)

#8. 평가함수

In [ ]:
def evaluate(model, data, top_k=10):
    model.eval()
    HR, NDCG = 0, 0

    with torch.no_grad():
        for user_id, (seq, gt) in enumerate(data):
            gt_idx  = gt[0]
            seq_set = set(seq)

            negs = []
            while len(negs) < NEG_EVAL:
                n = random.choice(all_item_list)
                if n not in seq_set and n != gt_idx:
                    negs.append(n)

            candidates = [gt_idx] + negs  # 101개

            u    = torch.LongTensor([user_id]).to(device)
            cand = torch.LongTensor([candidates]).to(device)  # (1, 101)

            u_emb    = model.user_emb(u)                                         # (1, D)
            cand_emb = model.item_emb(cand)                                      # (1, 101, D)
            scores   = torch.bmm(cand_emb, u_emb.unsqueeze(-1)).squeeze()        # (101,)

            rank = (scores[1:] > scores[0]).sum().item()

            if rank < top_k:
                HR   += 1
                NDCG += 1 / math.log2(rank + 2)

    n = len(data)
    return HR / n, NDCG / n

#9. 학습

In [ ]:
best_val_ndcg = 0.0
best_model    = model.state_dict()
counter       = 0

print("\n══════ MF 학습 시작 ══════")
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for user, pos, neg in loader:
        user = user.squeeze(1).to(device)
        pos  = pos.squeeze(1).to(device)
        neg  = neg.squeeze(1).to(device)

        optimizer.zero_grad()
        pos_score = model(user, pos)
        neg_score = model(user, neg)

        loss = (
            F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score)) +
            F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
        )
        # 원래 MF는 MSE를 쓰는데 우리는 평점 데이터가 아니라 "봤다/안봤다" 로 바꿔 쓰기때문에 이진 분류 문제로 바꿔버린거라서 BCE를 사용했다.
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    elapsed  = time.time() - t0
    avg_loss = total_loss / len(loader)

    val_hr, val_ndcg = evaluate(model, val_data)
    print(f"Epoch {epoch:3d}/{epochs}  Loss: {avg_loss:.4f}  "
          f"HR@10: {val_hr:.4f}  NDCG@10: {val_ndcg:.4f}  ({elapsed:.1f}s)")

    if val_ndcg > best_val_ndcg:
        best_val_ndcg = val_ndcg
        counter       = 0
        best_model    = model.state_dict()
        print(f"  ★ Best model saved (epoch {epoch})")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping!")
        break

        #MF는 그냥 유저벡터 × 아이템벡터 내적


══════ MF 학습 시작 ══════
Epoch   1/30  Loss: 3.9110  HR@10: 0.1558  NDCG@10: 0.0817  (83.1s)
  ★ Best model saved (epoch 1)
Epoch   2/30  Loss: 1.2881  HR@10: 0.4376  NDCG@10: 0.2411  (83.6s)
  ★ Best model saved (epoch 2)
Epoch   3/30  Loss: 1.1519  HR@10: 0.4291  NDCG@10: 0.2380  (83.2s)
Epoch   4/30  Loss: 1.1516  HR@10: 0.4291  NDCG@10: 0.2348  (81.9s)
Epoch   5/30  Loss: 1.1514  HR@10: 0.4389  NDCG@10: 0.2392  (81.1s)
Epoch   6/30  Loss: 1.1508  HR@10: 0.4316  NDCG@10: 0.2359  (81.7s)
Epoch   7/30  Loss: 1.1506  HR@10: 0.4326  NDCG@10: 0.2377  (81.2s)
Epoch   8/30  Loss: 1.1503  HR@10: 0.4371  NDCG@10: 0.2414  (81.9s)
  ★ Best model saved (epoch 8)
Epoch   9/30  Loss: 1.1503  HR@10: 0.4356  NDCG@10: 0.2396  (82.6s)
Epoch  10/30  Loss: 1.1500  HR@10: 0.4348  NDCG@10: 0.2396  (82.4s)
Epoch  11/30  Loss: 1.1499  HR@10: 0.4356  NDCG@10: 0.2411  (81.3s)
Epoch  12/30  Loss: 1.1496  HR@10: 0.4364  NDCG@10: 0.2427  (81.2s)
  ★ Best model saved (epoch 12)
Epoch  13/30  Loss: 1.1497  HR@10: 

#10. 최종 테스트

In [ ]:
print("\n──── 최종 테스트 평가 ────")
model.load_state_dict(best_model)
test_hr, test_ndcg = evaluate(model, test_data)
print(f"Test HR@10:   {test_hr:.4f}")
print(f"Test NDCG@10: {test_ndcg:.4f}")


──── 최종 테스트 평가 ────
Test HR@10:   0.4422
Test NDCG@10: 0.2399
